## Auto-TTL demo

This notebook uses SQL-only cells to create a small Unity Catalog managed Delta table in [workspace.default](#schema) with Auto-TTL enabled.

* `DELETE ROWS 1 DAYS AFTER created_at` is included directly in the `CREATE TABLE` statement.
* One row is inserted as already expired.
* One row is inserted as still active.
* Auto-TTL cleanup is asynchronous, so eligibility is immediate but physical deletion is not guaranteed to happen right away.
* Each SQL command is placed in its own cell so the flow is easy to test step by step.

In [0]:
USE CATALOG workspace;

In [0]:
USE SCHEMA default;

In [0]:
DROP TABLE IF EXISTS auto_ttl_demo;

In [0]:
CREATE TABLE auto_ttl_demo (
  id INT,
  created_at TIMESTAMP_NTZ
)
DELETE ROWS 1 DAYS AFTER created_at;

In [0]:
INSERT INTO auto_ttl_demo
VALUES
  (1, current_timestamp())

In [0]:
SHOW TBLPROPERTIES workspace.default.auto_ttl_demo;

In [0]:
SELECT *
FROM workspace.default.auto_ttl_demo
ORDER BY id;

In [0]:
DESCRIBE HISTORY auto_ttl_demo;

In [0]:
SELECT
  *
FROM system.storage.predictive_optimization_operations_history
WHERE operation_type = 'DELETE';

In [0]:
SELECT *
FROM workspace.default.auto_ttl_demo
ORDER BY id;

### What to expect

* Right after setup, you should see one row with `expired_now = true` and one row with `expired_now = false`.
* The expired row is only **eligible** for deletion immediately.
* When background maintenance runs, the monitoring query should show a delete-related optimization event, and the expired row should disappear from the final table query.
* If nothing happens for a while, predictive optimization may not be enabled for this schema or account.
* The table definition itself now demonstrates that Auto-TTL can be declared directly in `CREATE TABLE`.